In [ ]:
from pathlib import Path
import os
import random
import re
import time

import pandas as pd
import requests
from dotenv import load_dotenv

In [ ]:
load_dotenv()
API_KEY = os.getenv("OPENAQ_API_KEY")
assert API_KEY, "Missing OPENAQ_API_KEY in .env"

RAW_DIR = Path("../data/raw")
LOCATIONS_FILE = RAW_DIR / "openaq_locations_near_skopje_all_columns.parquet"

YEARS = range(2016, 2026)
DATETIME_FROM = "2016-01-01T00:00:00Z"
DATETIME_TO = "2026-01-01T00:00:00Z"
LIMIT = 1000
LOCAL_TZ = "Europe/Skopje"

In [13]:
locations = pd.read_parquet(LOCATIONS_FILE)

rows = []
for _, loc in locations.iterrows():
    for sensor in loc["sensors"]:
        rows.append(
            {
                "location_id": loc["id"],
                "location": loc["name"],
                "sensor_id": sensor["id"],
                "parameter": sensor["parameter"]["name"],
            }
        )

sensors = pd.DataFrame(rows)
sensors

,location_id,location,sensor_id,parameter
0,2919,Gazi Baba,7407,co
1,2919,Gazi Baba,4274431,no
2,2919,Gazi Baba,6196,no2
3,2919,Gazi Baba,6402034,o3
4,2919,Gazi Baba,6395,pm10
5,2919,Gazi Baba,6402028,pm25
6,2919,Gazi Baba,6366,so2
7,2976,Lisice,7401,co
8,2976,Lisice,4274710,no
9,2976,Lisice,6307,no2


In [19]:
AIR_QUALITY_DIR = RAW_DIR / "openaq_hourly"
AIR_QUALITY_DIR.mkdir(parents=True, exist_ok=True)

def fetch_sensor_hours_year(sensor_id, year):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"

    records = []
    page = 1
    max_attempts = 6

    while True:
        params = {
            "datetime_from": f"{year}-01-01T00:00:00Z",
            "datetime_to": f"{year + 1}-01-01T00:00:00Z",
            "limit": 1000,
            "page": page,
        }

        response = None

        for attempt in range(1, max_attempts + 1):
            try:
                response = requests.get(
                    url,
                    headers={"X-API-Key": API_KEY},
                    params=params,
                    timeout=(30, 120),
                )

                if response.status_code == 429:
                    retry_after = response.headers.get("Retry-After")
                    wait_seconds = (
                        int(retry_after)
                        if retry_after and retry_after.isdigit()
                        else min(60 * (2 ** (attempt - 1)), 900)
                        + random.randint(0, 20)
                    )

                    print(
                        f"      rate limited — waiting "
                        f"{wait_seconds} seconds"
                    )
                    time.sleep(wait_seconds)
                    continue

                response.raise_for_status()
                break

            except requests.exceptions.Timeout:
                wait_seconds = 20 * attempt
                print(
                    f"      timeout — waiting {wait_seconds} seconds"
                )
                time.sleep(wait_seconds)

            except requests.exceptions.RequestException as error:
                wait_seconds = 20 * attempt
                print(
                    f"      request error: {error} — "
                    f"waiting {wait_seconds} seconds"
                )
                time.sleep(wait_seconds)

        if response is None or response.status_code != 200:
            raise RuntimeError(
                f"Sensor {sensor_id}, year {year}, page {page} failed."
            )

        payload = response.json()
        rows = payload.get("results", [])

        if not rows:
            break

        records.extend(rows)

        if len(rows) < 1000:
            break

        page += 1
        time.sleep(1)

    if not records:
        return pd.DataFrame()

    return pd.json_normalize(records)

In [ ]:
download_log = []

for row in sensors.itertuples(index=False):
    location_id = int(row.location_id)
    location_name = str(row.location)
    sensor_id = int(row.sensor_id)
    parameter = str(row.parameter)

    location_slug = re.sub(
        r"[^a-z0-9]+",
        "_",
        location_name.lower()
    ).strip("_")

    sensor_dir = (
        AIR_QUALITY_DIR
        / f"location_{location_id}_{location_slug}"
        / f"sensor_{sensor_id}_{parameter}"
    )

    sensor_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"\n=== {location_name} | "
        f"{parameter} | sensor {sensor_id} ==="
    )

    for year in YEARS:
        output_file = sensor_dir / f"{year}.parquet"

        if output_file.exists():
            existing_df = pd.read_parquet(output_file)

            print(
                f"  {year}: already saved "
                f"({len(existing_df):,} rows)"
            )

            download_log.append({
                "location_id": location_id,
                "location_name": location_name,
                "sensor_id": sensor_id,
                "parameter": parameter,
                "year": year,
                "status": "already_saved",
                "rows": len(existing_df),
                "file": str(output_file),
            })

            continue

        print(f"  {year}: downloading...")

        try:
            year_df = fetch_sensor_hours_year(
                sensor_id=sensor_id,
                year=year,
            )

            if year_df.empty:
                print(f"  {year}: no data")

                download_log.append({
                    "location_id": location_id,
                    "location_name": location_name,
                    "sensor_id": sensor_id,
                    "parameter": parameter,
                    "year": year,
                    "status": "no_data",
                    "rows": 0,
                    "file": str(output_file),
                })

                time.sleep(2)
                continue

            year_df["source_location_id"] = location_id
            year_df["source_location_name"] = location_name
            year_df["source_sensor_id"] = sensor_id
            year_df["source_parameter"] = parameter

            year_df.to_parquet(output_file, index=False)

            print(f"  {year}: saved {len(year_df):,} rows")

            download_log.append({
                "location_id": location_id,
                "location_name": location_name,
                "sensor_id": sensor_id,
                "parameter": parameter,
                "year": year,
                "status": "saved",
                "rows": len(year_df),
                "file": str(output_file),
            })

            time.sleep(3)

        except Exception as error:
            print(f"  {year}: FAILED — {error}")

            download_log.append({
                "location_id": location_id,
                "location_name": location_name,
                "sensor_id": sensor_id,
                "parameter": parameter,
                "year": year,
                "status": "failed",
                "rows": None,
                "file": str(output_file),
                "error": str(error),
            })

            time.sleep(30)

download_log_df = pd.DataFrame(download_log)

download_log_df.to_parquet(
    AIR_QUALITY_DIR / "download_log.parquet",
    index=False,
)

display(download_log_df)